# TP2 â€” Retrieval-Augmented Generation (Part 2)
## From Retriever to Answering Agent â€” Evaluation with RAGAS

**Module 2IAiasd 9.7 â€” Agentic AI Â· IMT Mines AlÃ¨s**  
**Duration:** 2 h  
**Prerequisites:** TP2 Part 1 completed â€” `chroma_db/` folder must be in the same directory as this notebook.

---

### What you will build today

```
  Part 1 (done)                   Part 2 (today)
  â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€                   â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
  PDF â†’ chunks â†’ ChromaDB         ChromaDB â†’ retriever
                                       â”‚
                                       â–¼
                               Section 1: RetrievalQA chain (classic RAG)
                               Section 2: Hybrid search  (BM25 + vectors)
                               Section 3: Agentic RAG    (retrieve as @tool)
                               Section 4: RAGAS evaluation
                               Section 5: Classic RAG vs Agentic RAG
```

---

> **Important:** Run Part 1 first to generate `chroma_db/`.  
> If you skipped Part 1, set `REBUILD_INDEX = True` in Section 0 â€” the notebook will re-index automatically.

## Setup â€” Install dependencies

In [ ]:
%pip install -q \
    langchain \
    langchain-ollama \
    langchain-community \
    langchain-chroma \
    langchain-experimental \
    pypdf \
    chromadb \
    rank-bm25 \
    ragas

print("âœ… Dependencies installed")

## Imports

In [ ]:
import textwrap
import shutil
from pathlib import Path

# LangChain â€” core
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# LangChain â€” retrieval
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# LangChain â€” tools
from langchain_core.tools import tool

# Utilities
import numpy as np

print("âœ… Imports OK")

---
## Section 0 â€” Reload ChromaDB from Part 1

We reload the two collections built in Part 1 without re-embedding.

In [ ]:
# â”€â”€ CONFIGURE THIS â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
PDF_PATH    = "rapport_stage_2A.pdf"
CHROMA_DIR  = "./chroma_db"
EMBED_MODEL = "nomic-embed-text"
LLM_MODEL   = "qwen2.5:3b"
REBUILD_INDEX = False   # set True if you skipped Part 1
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

embeddings = OllamaEmbeddings(model=EMBED_MODEL)
llm        = ChatOllama(model=LLM_MODEL, temperature=0)

print(f"âœ… Embedding model : {EMBED_MODEL}")
print(f"âœ… LLM             : {LLM_MODEL}")

In [ ]:
def build_index_from_pdf(pdf_path: str, chroma_dir: str) -> None:
    """Re-index the PDF if chroma_db/ is missing (fallback for Part-1 skippers)."""
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    if Path(pdf_path).exists():
        pages = PyPDFLoader(pdf_path).load()
    else:
        # Minimal sample fallback
        SAMPLE = (
            "Chapter 1 â€” Introduction to Machine Learning. "
            "Machine learning enables computers to learn from data. "
            "Chapter 2 â€” Neural Networks. "
            "A neural network is composed of layers of neurons. "
            "Chapter 3 â€” Project Results. "
            "Our project achieved 87% accuracy using a fine-tuned BERT model. "
            "The main challenge was class imbalance, addressed with SMOTE."
        )
        pages = [Document(page_content=SAMPLE, metadata={"source": "sample", "page": 0})]

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks   = splitter.split_documents(pages)

    if Path(chroma_dir).exists():
        shutil.rmtree(chroma_dir)

    Chroma.from_documents(
        documents=chunks, embedding=embeddings,
        collection_name="rapport_recursive", persist_directory=chroma_dir,
    )
    print(f"âœ… Re-indexed {len(chunks)} chunks into '{chroma_dir}'")


if REBUILD_INDEX or not Path(CHROMA_DIR).exists():
    print("âš™ï¸  chroma_db/ not found â€” rebuilding index...")
    build_index_from_pdf(PDF_PATH, CHROMA_DIR)

# Load the recursive collection (main collection used throughout Part 2)
vectorstore = Chroma(
    collection_name="rapport_recursive",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)
n = vectorstore._collection.count()
print(f"âœ… Loaded 'rapport_recursive' â€” {n} vectors")

---
## Section 1 â€” Classic RAG: RetrievalQA Chain

Classic RAG = **one shot**: embed query â†’ retrieve top-k chunks â†’ prompt LLM with chunks as context â†’ answer.

```
  question
     â”‚
     â”œâ”€â”€â–º embed â”€â”€â–º ChromaDB â”€â”€â–º top-k chunks
     â”‚                                 â”‚
     â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜
                        â”‚
                        â–¼
               RAG prompt template
                        â”‚
                        â–¼
                  LLM (qwen2.5:3b)
                        â”‚
                        â–¼
                      answer
```

### 1.1 Build the RAG chain with LCEL

In [ ]:
# â”€â”€ Retriever â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# â”€â”€ RAG prompt â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
RAG_TEMPLATE = """\
You are an assistant that answers questions strictly based on the provided context.
If the answer is not in the context, say "I don't know â€” the document does not mention this."
Do not use any knowledge outside the context.

Context:
{context}

Question: {question}

Answer:"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

# â”€â”€ Helper: format retrieved documents into a single string â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(
        f"[Source: page {d.metadata.get('page','?')}]\n{d.page_content}"
        for d in docs
    )

# â”€â”€ LCEL chain â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# RunnablePassthrough keeps the original question available after retrieval.
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("âœ… Classic RAG chain ready")

### 1.2 Ask a question

In [ ]:
# â”€â”€ CONFIGURE THIS â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
QUESTION = "What were the main results of the internship project?"
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

print(f"Question: {QUESTION}")
print("â”€" * 60)

answer_classic = rag_chain.invoke(QUESTION)

print("Answer (Classic RAG):")
print(textwrap.fill(answer_classic, width=72))

### 1.3 Inspect what the retriever returned

Understanding **which** chunks the LLM received is essential for debugging.

In [ ]:
retrieved_docs = retriever.invoke(QUESTION)

print(f"Chunks sent to the LLM for: '{QUESTION}'")
print("â”€" * 60)
for i, doc in enumerate(retrieved_docs):
    print(f"\n[Chunk {i+1}] page={doc.metadata.get('page','?')} | {len(doc.page_content)} chars")
    print(textwrap.fill(doc.page_content[:300], width=70))

---
## Section 2 â€” Hybrid Search: BM25 + Vectors

Pure vector search can miss documents that share exact keywords with the query
(e.g. proper nouns, acronyms, technical terms).  
**BM25** (a keyword ranking algorithm) catches those.  
`EnsembleRetriever` merges both rankings with **Reciprocal Rank Fusion (RRF)**.

```
  question
     â”œâ”€â”€â–º BM25 (keyword)   â”€â”€â–º  ranked list A
     â”‚                                â”‚
     â”œâ”€â”€â–º ChromaDB (vector) â”€â”€â–º  ranked list B
     â”‚                                â”‚
     â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€ RRF merge â”€â”€â”€â”€â”€â”€â”€â”˜
                        â”‚
                        â–¼
                   top-k fused results
```

### 2.1 Build the hybrid retriever

In [ ]:
# Retrieve all chunks from ChromaDB to build the BM25 index
all_docs_data = vectorstore._collection.get(include=["documents", "metadatas"])
all_chunks = [
    Document(page_content=text, metadata=meta)
    for text, meta in zip(all_docs_data["documents"], all_docs_data["metadatas"])
]
print(f"Building BM25 index on {len(all_chunks)} chunks...")

# BM25 retriever â€” keyword-based
bm25_retriever = BM25Retriever.from_documents(all_chunks, k=4)

# Vector retriever â€” semantic
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Ensemble: 50% BM25 + 50% vector  (weights must sum to 1)
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)

print("âœ… Hybrid retriever ready (BM25 50% + vector 50%)")

### 2.2 Compare pure-vector vs hybrid retrieval

In [ ]:
def show_top_chunks(retriever, query: str, label: str, k: int = 3) -> list[Document]:
    docs = retriever.invoke(query)[:k]
    print(f"\n{'â”€'*60}")
    print(f"  {label}  |  query: '{query}'")
    print(f"{'â”€'*60}")
    for i, d in enumerate(docs):
        print(f"  [{i+1}] page={d.metadata.get('page','?')}  {d.page_content[:120].strip()}...")
    return docs


HYBRID_QUERY = "SMOTE accuracy BERT classification"  # keyword-heavy query

docs_vector = show_top_chunks(vector_retriever, HYBRID_QUERY, "VECTOR")
docs_hybrid = show_top_chunks(hybrid_retriever, HYBRID_QUERY, "HYBRID")

print()
print("Observation: hybrid retrieval often surfaces keyword-exact matches")
print("that pure vector search would rank lower.")

### 2.3 Hybrid RAG chain

In [ ]:
hybrid_rag_chain = (
    {"context": hybrid_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

answer_hybrid = hybrid_rag_chain.invoke(QUESTION)

print(f"Question: {QUESTION}")
print("â”€" * 60)
print("Answer (Hybrid RAG):")
print(textwrap.fill(answer_hybrid, width=72))

### 2.4 âœï¸ Exercise â€” Tune the ensemble weights

Try `weights=[0.3, 0.7]` (more vector) and `weights=[0.7, 0.3]` (more BM25).  
Run the same query and compare which configuration retrieves the most relevant chunk first.

**Question:** For a query that uses exact terms from the document (e.g. a method name),
which weight configuration performs better?

In [ ]:
# â”€â”€ YOUR CODE HERE â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 1. Build two EnsembleRetrievers with different weights
# 2. Run the same keyword-heavy query on both
# 3. Compare which top-1 result is more relevant

pass
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

---
## Section 3 â€” Agentic RAG: `retrieve` as a `@tool`

In classic RAG the retrieval is **hard-wired**: the pipeline always retrieves before answering,
even when the question is trivial.  
In **Agentic RAG** the LLM **decides** when to call the retriever:
it can retrieve once, retrieve again with a refined query, or answer directly from prior context.

This is exactly the ReAct loop from TP1 â€” with `retrieve` as one of the available tools.

```
  question
     â”‚
     â–¼
  LLM â”€â”€â–º Thought: I need information about X
       â”€â”€â–º Action: retrieve(query="X")
       â—„â”€â”€ Observation: [chunk 1, chunk 2, ...]
       â”€â”€â–º Thought: I have enough context now
       â”€â”€â–º Answer: ...
```

### 3.1 Define the `retrieve` tool

In [ ]:
@tool
def retrieve(query: str) -> str:
    """
    Search the internship report for information relevant to the query.
    Returns the top-4 most relevant passages from the document.
    Use this tool whenever you need information from the report to answer a question.
    """
    docs = vectorstore.as_retriever(search_kwargs={"k": 4}).invoke(query)
    if not docs:
        return "No relevant passages found in the document."
    return "\n\n".join(
        f"[page {d.metadata.get('page','?')}] {d.page_content}"
        for d in docs
    )


# Quick test
test_result = retrieve.invoke("What accuracy was achieved?")
print("Tool test â€” first 300 chars of result:")
print(test_result[:300])
print("...")
print()
print("âœ… retrieve tool ready")

### 3.2 Build the Agentic RAG loop

We reuse the same ReAct loop structure as TP1 â€” only the tool changes.

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage
import json

# Bind the retrieve tool to the LLM
agentic_llm = llm.bind_tools([retrieve])

AGENTIC_SYSTEM = """You are a helpful assistant that answers questions about an internship report.
You have access to a retrieve tool that searches the report.
Always use the retrieve tool at least once before answering.
Call one tool at a time. When you have enough context, give a clear final answer."""


def agentic_rag(question: str, max_steps: int = 6, verbose: bool = True) -> str:
    """Run an Agentic RAG loop for one question."""
    messages = [
        SystemMessage(AGENTIC_SYSTEM),
        HumanMessage(question),
    ]

    for step in range(max_steps):
        response = agentic_llm.invoke(messages)
        messages.append(response)

        # â”€â”€ No tool call â†’ final answer â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        if not response.tool_calls:
            if verbose:
                print(f"  [step {step+1}] Final answer reached.")
            return response.content

        # â”€â”€ Tool call â†’ execute and feed result back â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        # Enforce one tool at a time (same guard as TP1)
        tool_call = response.tool_calls[0]
        if verbose:
            print(f"  [step {step+1}] Tool call: {tool_call['name']}({tool_call['args']})")

        # Execute the tool
        tool_result = retrieve.invoke(tool_call["args"]["query"])

        messages.append(
            ToolMessage(
                content=tool_result,
                tool_call_id=tool_call["id"],
            )
        )

        if verbose:
            print(f"         â†’ {tool_result[:100].strip()}...")

    return "[max steps reached without final answer]"


print("âœ… Agentic RAG loop ready")

### 3.3 Run the Agentic RAG

In [ ]:
print(f"Question: {QUESTION}")
print("â”€" * 60)
print("Agentic RAG trace:")
answer_agentic = agentic_rag(QUESTION, verbose=True)
print()
print("â”€" * 60)
print("Answer (Agentic RAG):")
print(textwrap.fill(answer_agentic, width=72))

### 3.4 âœï¸ Exercise â€” Multi-step retrieval

Ask a question that requires **two separate retrieval steps**:
first retrieve the method used, then retrieve the results for that method.

Example: *"Compare the accuracy of the main model with the baseline. Which performed better and by how much?"*

**Observe:** Does the agent issue two `retrieve` calls with different queries,
or does it answer in one step?

In [ ]:
# â”€â”€ YOUR CODE HERE â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 1. Write a question that requires two retrieval steps
# 2. Call agentic_rag() with verbose=True
# 3. Count the number of retrieve calls in the trace

MULTI_STEP_QUESTION = "Compare the accuracy of the main model with the baseline."

# TODO: call agentic_rag() and print the answer
pass
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

---
## Section 4 â€” Evaluation with RAGAS

RAGAS measures RAG quality **without human labelling** using three metrics:

| Metric | Question answered | Range |
|---|---|---|
| **Faithfulness** | Is the answer supported by the retrieved context? | 0â€“1 |
| **Context Precision** | Are the retrieved chunks actually relevant to the question? | 0â€“1 |
| **Answer Relevancy** | Does the answer address the question? | 0â€“1 |

> **Important:** RAGAS uses an LLM internally to judge each metric.  
> We configure it to use our local `qwen2.5:3b` via LangChain â€” no API key needed.

### 4.1 Build the evaluation dataset

In [ ]:
# â”€â”€ CONFIGURE THIS â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Write 4â€“6 questions about your internship report.
# Cover different sections: objective, methods, challenges, results.
# If using sample text, the questions below work out of the box.

EVAL_QUESTIONS = [
    "What was the main objective of the internship project?",
    "Which machine learning methods were used?",
    "What was the accuracy achieved on the test set?",
    "What was the main challenge and how was it addressed?",
    "What are the future directions mentioned in the report?",
]
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

print(f"Evaluation dataset: {len(EVAL_QUESTIONS)} questions")

In [ ]:
# Generate answers and retrieve contexts for each question
# We evaluate BOTH classic RAG and agentic RAG

print("Generating answers for classic RAG...")
classic_answers  = []
classic_contexts = []

for q in EVAL_QUESTIONS:
    docs    = retriever.invoke(q)
    context = [d.page_content for d in docs]
    answer  = rag_chain.invoke(q)
    classic_answers.append(answer)
    classic_contexts.append(context)
    print(f"  âœ“ {q[:60]}...")

print()
print("Generating answers for agentic RAG...")
agentic_answers  = []
agentic_contexts = []

for q in EVAL_QUESTIONS:
    # For RAGAS we need the contexts too â€” capture them from a retrieval call
    docs    = retriever.invoke(q)
    context = [d.page_content for d in docs]
    answer  = agentic_rag(q, verbose=False)
    agentic_answers.append(answer)
    agentic_contexts.append(context)
    print(f"  âœ“ {q[:60]}...")

print("\nâœ… Answers generated")

### 4.2 Run RAGAS evaluation

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Configure RAGAS to use our local LLM and embeddings
ragas_llm   = LangchainLLMWrapper(llm)
ragas_emb   = LangchainEmbeddingsWrapper(embeddings)

metrics = [faithfulness, answer_relevancy, context_precision]
for m in metrics:
    m.llm = ragas_llm
    if hasattr(m, "embeddings"):
        m.embeddings = ragas_emb

print("âœ… RAGAS configured with local LLM")

In [ ]:
# Evaluate classic RAG
print("Evaluating classic RAG (this may take a few minutes)...")
classic_dataset = Dataset.from_dict({
    "question":  EVAL_QUESTIONS,
    "answer":    classic_answers,
    "contexts":  classic_contexts,
})
classic_results = evaluate(classic_dataset, metrics=metrics)
classic_df = classic_results.to_pandas()

print("\nðŸ“Š Classic RAG â€” per-question scores:")
print(classic_df[["question", "faithfulness", "answer_relevancy", "context_precision"]]
      .to_string(index=False))

In [ ]:
# Evaluate agentic RAG
print("Evaluating agentic RAG (this may take a few minutes)...")
agentic_dataset = Dataset.from_dict({
    "question":  EVAL_QUESTIONS,
    "answer":    agentic_answers,
    "contexts":  agentic_contexts,
})
agentic_results = evaluate(agentic_dataset, metrics=metrics)
agentic_df = agentic_results.to_pandas()

print("\nðŸ“Š Agentic RAG â€” per-question scores:")
print(agentic_df[["question", "faithfulness", "answer_relevancy", "context_precision"]]
      .to_string(index=False))

---
## Section 5 â€” Classic RAG vs Agentic RAG: Final Comparison

### 5.1 Summary table

In [ ]:
import warnings
warnings.filterwarnings("ignore")

metric_cols = ["faithfulness", "answer_relevancy", "context_precision"]

print("\n" + "â•" * 60)
print("  RAGAS COMPARISON SUMMARY")
print("â•" * 60)
print(f"  {'Metric':<25}  {'Classic RAG':>12}  {'Agentic RAG':>12}")
print("  " + "â”€" * 54)

for col in metric_cols:
    c_mean = classic_df[col].mean()
    a_mean = agentic_df[col].mean()
    winner = "â—„" if a_mean > c_mean else ""
    print(f"  {col:<25}  {c_mean:>12.3f}  {a_mean:>12.3f}  {winner}")

print("â•" * 60)
print("  â—„ marks the better score for each metric")

### 5.2 Per-question comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x     = np.arange(len(metric_cols))
width = 0.35

classic_means = [classic_df[m].mean() for m in metric_cols]
agentic_means = [agentic_df[m].mean() for m in metric_cols]

fig, ax = plt.subplots(figsize=(8, 4))
bars_c = ax.bar(x - width/2, classic_means, width, label="Classic RAG",
                color="#0097A7", alpha=0.85)
bars_a = ax.bar(x + width/2, agentic_means, width, label="Agentic RAG",
                color="#F24960", alpha=0.85)

ax.set_ylabel("RAGAS score")
ax.set_title("Classic RAG vs Agentic RAG â€” RAGAS metrics")
ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", "\n") for m in metric_cols])
ax.set_ylim(0, 1.1)
ax.legend()
ax.bar_label(bars_c, fmt="%.2f", padding=2, fontsize=8)
ax.bar_label(bars_a, fmt="%.2f", padding=2, fontsize=8)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("ragas_comparison.png", dpi=150)
plt.show()
print("Saved: ragas_comparison.png")

### 5.3 When to use which approach?

| | Classic RAG | Agentic RAG |
|---|---|---|
| **Retrieval** | Fixed: always k chunks, one shot | Flexible: the LLM decides when and what to retrieve |
| **Multi-step** | No â€” one retrieval per question | Yes â€” can retrieve multiple times with refined queries |
| **Latency** | Lower â€” one LLM call | Higher â€” multiple LLM calls |
| **Faithfulness** | Bounded by quality of first retrieval | Can improve by re-querying |
| **Complexity** | Simple chain | Requires a loop + tool binding |
| **Best for** | Simple factual questions | Complex questions requiring synthesis |

### 5.4 âœï¸ Exercise â€” Identify failure cases

Find one question where **Classic RAG fails** but **Agentic RAG succeeds** (or vice versa).  
Explain why by looking at the RAGAS per-question scores and the retrieved chunks.

In [ ]:
# â”€â”€ YOUR CODE HERE â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 1. Find a question where classic_df["faithfulness"] < agentic_df["faithfulness"]
#    (or any metric where the gap is large)
# 2. Print the classic answer and the agentic answer for that question
# 3. Print the retrieved chunks for that question
# 4. Explain the difference in 2-3 sentences

pass
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

---
## Checkpoint â€” What you have built

```
  ChromaDB (from Part 1)
       â”‚
       â”œâ”€â”€â–º Classic RAG chain (LCEL)              âœ…
       â”‚         â”‚
       â”‚         â–¼
       â”‚    answer (one-shot retrieval)
       â”‚
       â”œâ”€â”€â–º Hybrid retriever (BM25 + vector)       âœ…
       â”‚         â”‚
       â”‚         â–¼
       â”‚    better keyword recall
       â”‚
       â””â”€â”€â–º Agentic RAG (retrieve as @tool)        âœ…
                 â”‚
                 â–¼
            multi-step reasoning loop
                 â”‚
                 â–¼
            RAGAS evaluation                       âœ…
            (faithfulness Â· context precision Â· answer relevancy)
```

### In TP3 (CrewAI) you will add:

```
  Agentic RAG tool
       â”‚
       â–¼
  CrewAI multi-agent system
  (Researcher agent + Writer agent + Reviewer agent)
       â”‚
       â–¼
  Structured report generated collaboratively
```